# 02. M2 전처리 – raw 데이터 6개 점검
각 셀을 위에서부터 `Shift+Enter`로 한 칸씩 실행하세요. 이 노트북은 파일을 **읽기만** 하고 raw는 수정하지 않습니다.

## 1. raw 파일 찾기

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent          # 노트북이 하위 폴더(notebooks/)에 있을 때
RAW = ROOT / "data" / "raw"
files = sorted(p for p in RAW.glob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls"))
print("raw 폴더:", RAW)
for i, p in enumerate(files, 1):
    print(i, p.name)

## 2. CSV: 인코딩 자동 판별 + 크기·컬럼·결측·중복

In [ ]:
def read_csv_auto(p):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(p, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {p.name}")

dfs = {}
for p in files:
    if p.suffix.lower() != ".csv":
        continue
    df, enc = read_csv_auto(p)
    df.columns = df.columns.astype(str).str.replace(r"\s+", "", regex=True)
    dfs[p.name] = df
    miss = df.isna().sum()
    print(f"\n== {p.name} | {enc} | {df.shape[0]}행 x {df.shape[1]}열")
    print("컬럼:", list(df.columns))
    print("결측:", miss[miss > 0].to_dict() or "없음", "| 중복 행:", int(df.duplicated().sum()))

## 3. 엑셀: 모든 시트를 열어서 표인지 문서인지 확인
1번 파일은 첫 시트가 비어 있었으므로 시트 전체를 봅니다.

In [ ]:
for p in files:
    if p.suffix.lower() not in (".xlsx", ".xls"):
        continue
    sheets = pd.read_excel(p, sheet_name=None, header=None)
    print(f"\n== {p.name} | 시트 {len(sheets)}개")
    for name, df in sheets.items():
        filled = int(df.notna().sum().sum())
        print(f"  [{name}] {df.shape[0]}행 x {df.shape[1]}열, 값 있는 칸 {filled}개")
        for v in df.stack().head(6):
            print("     ·", str(v)[:90].replace("\n", " "))

## 4. 요약표 (노션에 붙여넣기용)

In [ ]:
rows = []
for name, df in dfs.items():
    num = df.select_dtypes("number")
    out = 0
    for c in num.columns:
        q1, q3 = num[c].quantile([0.25, 0.75]); iqr = q3 - q1
        out += int(((num[c] < q1 - 1.5*iqr) | (num[c] > q3 + 1.5*iqr)).sum())
    rows.append([name, df.shape[0], df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()), out])
summary = pd.DataFrame(rows, columns=["파일", "행", "열", "결측합", "중복", "이상치후보(IQR)"])
summary

## 5. 전처리 저장 (data/processed 7개 파일 생성)
raw는 수정하지 않고, 정리한 파일만 data/processed에 저장합니다.

In [ ]:
from pathlib import Path
import pandas as pd, re, shutil, datetime as dt

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed"
if OUT.exists() and any(OUT.iterdir()):
    bak = ROOT / "data" / f"processed_backup_{dt.datetime.now():%m%d_%H%M}"
    shutil.move(str(OUT), str(bak)); print("기존 processed 백업:", bak.name)
OUT.mkdir(parents=True, exist_ok=True)
files = sorted(p for p in RAW.glob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls"))
get = lambda k: next(p for p in files if p.name.startswith(k + "_"))

def csv_auto(p):
    for enc in ("utf-8-sig", "cp949"):
        try: return pd.read_csv(p, encoding=enc)
        except UnicodeDecodeError: continue

IND_MAP = {"전기가스증기수도사업": "전기·가스·증기·수도사업"}
def std_ind(s):
    if pd.isna(s): return s
    s = re.sub(r"\s+", "", str(s)); return IND_MAP.get(s, s)
def std_scale(s):
    if pd.isna(s): return s
    s = re.sub(r"근로자수|사고사망자수|사고재해자수|\s|,", "", str(s)).replace("~", "-")
    return re.sub(r"인-", "-", s)
def clean_text(s):
    if pd.isna(s): return s
    return re.sub(r"[ \t\u00a0]+", " ", str(s)).strip()
def short(p):
    s = re.sub(r"^\d+_(한국산업안전보건공단|고용노동부)_|_\d{8}$", "", p.stem)
    return f"{p.name.split('_')[0]}_{s.replace(' ', '_')}.csv"
def n_outliers(d):
    n = 0
    for c in d.select_dtypes("number").columns:
        q1, q3 = d[c].quantile([.25, .75]); i = q3 - q1
        n += int(((d[c] < q1 - 1.5 * i) | (d[c] > q3 + 1.5 * i)).sum())
    return n

report = []
def save(d, name, src):
    d.to_csv(OUT / name, index=False, encoding="utf-8-sig")
    report.append([name, src, d.shape[0], d.shape[1], int(d.isna().sum().sum()), int(d.duplicated().sum()), n_outliers(d)])

# --- 1번: SIF 아카이브 (사고 사례 = RAG 문서) ---
x1 = get("1")
mf = pd.read_excel(x1, sheet_name="아카이브(제조업등)", header=2, usecols="B:J")
mf.columns = [str(c).replace("\n", "").replace(" ", "") for c in mf.columns]
cs = pd.read_excel(x1, sheet_name="아카이브(건설업)", header=None, skiprows=4, usecols="B:J")
cs.columns = ["연번", "공종", "작업명", "단위작업명", "재해종류", "재해개요", "기인물", "재해유발요인", "위험성감소대책(예시)"]
for name, d, 업종 in [("1_SIF_제조업등.csv", mf, "제조업등"), ("1_SIF_건설업.csv", cs, "건설업")]:
    d = d.dropna(subset=["재해개요"]).reset_index(drop=True)
    for c in d.select_dtypes("object").columns: d[c] = d[c].map(clean_text)
    if "산재업종(중분류)" in d.columns:
        for c in ["산재업종(대분류)", "산재업종(중분류)", "산재업종(소분류)"]: d[c] = d[c].map(std_ind)
    else:
        d["산재업종(중분류)"] = "건설업"
    d["업종구분"] = 업종
    d["기인물_표준"] = d["기인물"].where(d["기인물"] != "분류 불가")
    d["재해연도"] = pd.to_numeric(d["재해개요"].str.extract(r"(20\d{2})년")[0], errors="coerce")
    d["재해개요중복"] = d["재해개요"].duplicated(keep=False)
    save(d, name, "1번 엑셀")

# --- 2~5번: 통계 CSV ---
for k in "2345":
    p = get(k); d = csv_auto(p)
    cols = []
    for c in d.columns:
        c0 = re.sub(r"\s+", "", str(c))
        cols.append(c0 if c0 in ("대업종", "구분") else std_scale(c0))
    d.columns = cols
    if "대업종" in d.columns: d["대업종"] = d["대업종"].map(clean_text)
    d["구분"] = d["구분"].map(std_scale if k == "3" else std_ind)
    for c in d.columns:
        if c in ("대업종", "구분") or d[c].dtype != object: continue
        t = pd.to_numeric(d[c].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
        if d[c].notna().sum() > 0 and t.notna().sum() >= 0.9 * d[c].notna().sum(): d[c] = t
    save(d, short(p), p.name[:12])

# --- 6번: 출처 목록 (텍스트 정리만) ---
p6 = get("6"); d6 = csv_auto(p6)
for c in d6.select_dtypes("object").columns: d6[c] = d6[c].map(clean_text)
save(d6, short(p6), "6번 출처목록")

summary = pd.DataFrame(report, columns=["저장 파일", "원본", "행", "열", "결측(NaN)", "중복행", "이상치후보(IQR)"])
print("저장 위치:", OUT)
summary

## 6. 저장 파일명 정리

In [ ]:
from pathlib import Path
import pandas as pd, unicodedata, re

ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "data" / "processed"

# 1) 파일명 정리: 한글 정규화 + 번호 중복 제거
for p in sorted(OUT.glob("*.csv")):
    s = unicodedata.normalize("NFC", p.stem)
    s = re.sub(r"^(\d+)_\1_", r"\1_", s)
    s = re.sub(r"(한국산업안전보건공단|고용노동부)_|_\d{8}$", "", s)
    new = OUT / f"{s}.csv"
    if new != p and not new.exists():
        p.rename(new)
print([p.name for p in sorted(OUT.glob("*.csv"))])

# 2) 5번 결측 14개의 위치
f5 = next(p for p in OUT.glob("5_*.csv"))
d5 = pd.read_csv(f5, encoding="utf-8-sig")
na = d5.isna()
print("\n[5번 결측 컬럼별]", na.sum()[na.sum() > 0].to_dict())
print(d5.loc[na.any(axis=1), ["대업종", "구분"]].assign(결측수=na.sum(axis=1)[na.any(axis=1)]).to_string())

# 3) 1번 연도 추출 실패 건
for f in sorted(OUT.glob("1_*.csv")):
    d = pd.read_csv(f, encoding="utf-8-sig")
    bad = d[d["재해연도"].isna()]
    print(f"\n[{f.name}] 연도 NaN {len(bad)}건")
    for t in bad["재해개요"].head(6): print("  ·", str(t)[:60].replace("\n", " "))

# 4) 이상치후보가 많은 컬럼 (상위 5)
for f in sorted(OUT.glob("[2-5]_*.csv")):
    d = pd.read_csv(f, encoding="utf-8-sig"); rows = []
    for c in d.select_dtypes("number").columns:
        q1, q3 = d[c].quantile([.25, .75]); i = q3 - q1
        n = int(((d[c] < q1 - 1.5*i) | (d[c] > q3 + 1.5*i)).sum())
        if n: rows.append((c, n))
    print(f"\n[{f.name}] 이상치후보 컬럼:", sorted(rows, key=lambda x: -x[1])[:5])

## 7. 연도 6건 보정 · 5번 결측 14칸 대조 · 이상치(IQR) 후보

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "processed").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "data" / "processed"

# 1) 재해연도 보정 (원문 오타 '2023월' 대응). 재해개요 원문은 그대로 둠
for f in sorted(OUT.glob("1_*.csv")):
    d = pd.read_csv(f, encoding="utf-8-sig")
    y = d["재해개요"].str.extract(r"(20\d{2})(?:년|월|\.)")[0].astype(float)
    fixed = d["재해연도"].isna() & y.notna()
    d.loc[fixed, "재해연도"] = y[fixed]
    d["재해연도"] = d["재해연도"].astype("Int64")
    d.to_csv(f, index=False, encoding="utf-8-sig")
    print(f"{f.name}: 보정 {int(fixed.sum())}건 | 남은 NaN {int(d['재해연도'].isna().sum())}")

# 2) 5번 결측 14개가 4번 사업장수 0과 일치하는지
d4 = pd.read_csv(next(OUT.glob("4_*.csv")), encoding="utf-8-sig")
d5 = pd.read_csv(next(OUT.glob("5_*.csv")), encoding="utf-8-sig")
size5 = [c for c in d5.columns if c not in ("대업종", "구분")]
rows = []
for _, r in d5.iterrows():
    for c in size5:
        if pd.isna(r[c]):
            col4 = c + "사업장수"
            v = d4.loc[d4["구분"] == r["구분"], col4].values[0] if col4 in d4.columns else "컬럼없음"
            rows.append((r["구분"], c, v))
chk = pd.DataFrame(rows, columns=["구분", "규모", "4번 사업장수"])
print("\n[5번 결측 14개 vs 4번 사업장수]")
print(chk.to_string())
print("사업장수가 0 또는 결측인 칸:", int(chk["4번 사업장수"].fillna(0).eq(0).sum()), "/", len(chk))

# 3) 5번 사망만인율 극단값 상위
num = d5[size5]
rows = []
for c in size5:
    q1, q3 = num[c].quantile([.25, .75]); i = q3 - q1
    for idx in num.index[(num[c] > q3 + 1.5 * i) | (num[c] < q1 - 1.5 * i)]:
        rows.append((c, d5.loc[idx, "구분"], num.loc[idx, c]))
o = pd.DataFrame(rows, columns=["규모", "구분", "사망만인율"])
print("\n[사망만인율 극단값 상위 10]")
print(o.sort_values("사망만인율", ascending=False).head(10).to_string())
print("이상치가 자주 걸린 업종:", o["구분"].value_counts().head(5).to_dict())